# trainer-class-skeleton — worked example 3: Trainer that logs sample predictions during validation

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `trainer-class-skeleton`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

A common extension to the base Trainer is to surface a few sample predictions alongside the loss during validation. The `validate` method can store `(logits, targets)` from the first batch of the validation set into `self.last_val_samples` for external inspection. This is useful for quick sanity checks — you can print or plot predictions without adding visualization code to the training loop.

## Worked solution

**Step 1 – New attribute `self.last_val_samples`.** Initialized to `None` in `__init__`, it gets overwritten each time `validate()` runs with the `(logits, targets)` from the first validation batch.

**Step 2 – `validate` captures one batch.** We add a `first_batch = True` flag. On the first batch we detach and clone the logits and targets, storing them as `self.last_val_samples`. Subsequent batches proceed normally (they only contribute to the loss average).

**Step 3 – Everything else unchanged.** The `fit` loop and `_step` are identical to the base skeleton. The sample logging is entirely inside `validate`, so no structural change is needed.

**Step 4 – Inspect after training.** After `fit` returns, the caller can do `trainer.last_val_samples` to inspect a batch of predictions. A real trainer might save these to disk or pass them to a wandb table.

In [ ]:
import torch as t
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

class WorkedTrainer3LogSamples:
    def __init__(self, model, optimizer, train_loader, val_loader, loss_fn):
        self.model = model
        self.optimizer = optimizer
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.loss_fn = loss_fn
        self.step = 0
        self.history = {'train_loss': [], 'val_loss': []}
        self.last_val_samples = None  # (logits, targets) from first val batch

    def _step(self, x, y):
        return self.loss_fn(self.model(x), y)

    def fit(self, n_epochs):
        for _ in range(n_epochs):
            self.model.train()
            for x, y in self.train_loader:
                loss = self._step(x, y)
                loss.backward()
                self.optimizer.step()
                self.optimizer.zero_grad()
                self.step += 1
                self.history['train_loss'].append(loss.item())
            self.validate()

    def validate(self):
        self.model.eval()
        total, count = 0.0, 0
        first_batch = True
        with t.inference_mode():
            for x, y in self.val_loader:
                logits = self.model(x)
                loss = self.loss_fn(logits, y)
                total += loss.item() * x.shape[0]
                count += x.shape[0]
                if first_batch:
                    self.last_val_samples = (
                        logits.detach().clone(),
                        y.detach().clone(),
                    )
                    first_batch = False
        self.history['val_loss'].append(total / count)

# Demo
t.manual_seed(7)
X = t.randn(50, 2)
Y = X[:, 0:1] + 0.5 * X[:, 1:2]
train_dl = DataLoader(TensorDataset(X[:40], Y[:40]), batch_size=8)
val_dl = DataLoader(TensorDataset(X[40:], Y[40:]), batch_size=10)
model = nn.Linear(2, 1)
trainer = WorkedTrainer3LogSamples(model, t.optim.Adam(model.parameters()),
                                    train_dl, val_dl, nn.MSELoss())
trainer.fit(2)
logits, targets = trainer.last_val_samples
print('sample logits shape:', logits.shape)
print('sample targets shape:', targets.shape)
print('val losses:', trainer.history['val_loss'])